In [0]:
source_path = "/Volumes/olist_catalog/volumes/blob_sources"

def run_autoloader(file_pattern, table_name):
    # Đưa checkpoint và schema vào làm thư mục con TRONG volume blob_sources
    checkpoint_path = f"/Volumes/olist_catalog/volumes/blob_sources/_checkpoints/{table_name}/_checkpoint"
    schema_path = f"/Volumes/olist_catalog/volumes/blob_sources/_checkpoints/{table_name}/_schema"
    
    target_table = f"olist_catalog.blob_bronze.{table_name}"
    
    print(f"Đang ingest dữ liệu vào bảng: {target_table} ...")
    
    # 1. ĐỌC: Cấu hình Auto Loader
    df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true") 
        .option("cloudFiles.schemaLocation", schema_path)
        .option("pathGlobFilter", file_pattern)        
        .load(source_path)
    )
    
    # 2. GHI: Lưu thành bảng Delta
    (df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True) 
        .toTable(target_table)
    )
    
    print(f"Hoàn tất setup luồng cho: {target_table}\n")

# Chạy lại luồng Ingest
run_autoloader("inventory_snapshot*.csv", "inventory_snapshot")
run_autoloader("marketing_campaign*.csv", "marketing_campaign")